In [ ]:
import os
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import userdata


login(userdata.get('HF_TOKEN'))

modelId = "rvinay73/gate-shape-compiler"

tokenizer = AutoTokenizer.from_pretrained(modelId, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(modelId, device_map="auto", torch_dtype="auto").eval()



In [17]:
def build_prompt(userInput: str) -> str:
    return (
        "### Instruction:\n"
        "Extract ALL gate-size connectivity rules from the input. "
        "Output ONLY valid JSON matching this schema: "
        "{\"rules\":[{\"nQubits\":int,\"shape\":string,\"edges\":[[int,int],...]},...]}.\n"
        "Use canonical node ids 0..k-1 for each rule.\n"
        "### Input:\n"
        f"{userInput}\n"
        "### Output:\n"
    )

userInput = "My 4 qubit gates should be in a line; My 3 qubit gates should be in a triangle"
prompt = build_prompt(userInput)

messages = [{"role": "user", "content": prompt}]

In [18]:
# Build inputs (this returns a BatchEncoding dict)
enc = tokenizer.apply_chat_template(
    conversation=messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
)

# Move to device (BatchEncoding supports .to)
enc = enc.to(model.device)

# Generate using dict unpacking (works reliably)
out = model.generate(**enc, max_new_tokens=80)

# Use tensor length WITHOUT .shape
promptLen = enc["input_ids"].size(-1)

response = tokenizer.decode(out[0][promptLen:], skip_special_tokens=True)
print(response)

{"rules":[{"nQubits":3,"shape":"triangle","edges":[[0,1],[0,2],[1,2]]},{"nQubits":4,"shape":"line","edges":[[0,1],[1,2],[2,3]]}]}
